## Step 03 — Translate transcriptions to English

This notebook reads the transcriptions CSV produced by notebook `01` and translates any non-English transcripts into English using Google Translate (via the [`deep-translator`](https://github.com/nidhaloff/deep-translator) library).

Why this step is needed: Whisper detects and transcribes audio in whatever language is spoken, so a dataset of TikTok videos from multiple accounts is likely to contain many languages. The clustering step in notebook `03` works on the text itself, so all transcripts need to be in the same language first.

How it works:

- English transcripts are passed through unchanged
- Non-English transcripts are sent to Google Translate with the identified language code from Step 02 as the source
- Each translation is retried once if it fails
- Progress is checkpointed to the output CSV every `SAVE_EVERY` rows so you can safely interrupt and resume
- After translation, the results are merged with the original video metadata and saved as a combined CSV

**Input:** `output/transcriptions.csv`  
**Outputs:**

- `output/transcriptions_translated.csv` — transcriptions with an added `transcript_en` column
- `output/transcriptions_and_translations.csv` — translations joined with original video metadata table


In [ ]:
# INSTALL REQUIREMENTS (one time only):

# (locally)
# %pip install -r ../requirements.txt

# (on google colab)
# !pip install deep-translator


import pandas as pd
import time
from tqdm import tqdm
from deep_translator import GoogleTranslator
import os

# (on google colab)
# from google.colab import drive
# drive.mount('/content/drive')

### Translation Script

Scroll through the section headers inside the cell for a step-by-step breakdown.


In [ ]:
# =============================
# 1. SET TRANSLATION CONFIGS
# =============================

# In google drive: 
# PATH_PREFIX = "/content/drive/MyDrive/tiktok-content-reupload-tracker"

# Locally:
PATH_PREFIX = ".."

INPUT_CSV = f"{PATH_PREFIX}/output/transcriptions.csv"
OUTPUT_CSV = f"{PATH_PREFIX}/output/transcriptions_translated.csv"

SAVE_EVERY = 100
SLEEP_BETWEEN_REQUESTS = 0.5

# =============================
# 2. LOAD DATA
# =============================
df = pd.read_csv(INPUT_CSV)

if "transcript" not in df.columns or "language" not in df.columns:
    raise ValueError("CSV must contain 'transcript' and 'language' columns")

if "transcript_en" not in df.columns:
    df["transcript_en"] = None

# =============================
# 3. RESUME TRANSLATION FROM LAST SAVED 
# =============================
if os.path.exists(OUTPUT_CSV):
    existing = pd.read_csv(OUTPUT_CSV)
    if "transcript_en" in existing.columns:
        df["transcript_en"] = existing["transcript_en"]
    print("Resuming from existing output...")

# =============================
# 4. DEFINE TRANSLATION FUNCTION
# =============================
def translate(text, source_lang):
    if not isinstance(text, str) or not text.strip():
        return None

    try:
        translated = GoogleTranslator(
            source="auto", # Use GoogleTranslator's auto-detection for source language
            target="en"
        ).translate(text)

        return translated

    except Exception as e:
        print(f"Error [{source_lang}]:", e)
        return None  # leave blank on failure

# =============================
# 5. RUN TRANSLATION
# =============================
for idx, row in tqdm(df.iterrows(), total=len(df)):

    # skip already done
    if pd.notna(row.get("transcript_en")):
        continue

    # skip English
    if row["language"] == "en":
        df.at[idx, "transcript_en"] = row["transcript"]
        continue

    text = row["transcript"]
    lang = row["language"]

    translated = None

    # retry once
    for attempt in range(2):
        translated = translate(text, lang)
        if translated:
            break
        time.sleep(1)

    df.at[idx, "transcript_en"] = translated

    # checkpoint save
    if idx % SAVE_EVERY == 0:
        df.to_csv(OUTPUT_CSV, index=False)

    time.sleep(SLEEP_BETWEEN_REQUESTS)

# =============================
# 6. SAVE RESULTS
# =============================
df.to_csv(OUTPUT_CSV, index=False)

print("✅ Done:", OUTPUT_CSV)

### Merging Translations With Original Metadata

Join the translated transcripts back onto the original metadata CSV so that downstream notebooks have both the transcript text and all video metadata (e.g. author, post date, video URL) in one place.

The join key is meta_id (metadata) <-> file_name (transcription). file_name is the yt-dlp output filename which is the TikTok video ID, so we strip the file extension before joining.


In [ ]:
METADATA_FILE_PATH = "videofiles-extracted/video_metadata.csv"

videos = pd.read_csv(f"{PATH_PREFIX}/output/{METADATA_FILE_PATH}")
videos['meta_id'] = videos['meta_id'].astype(str)  # ensure string type for reliable join

translations = pd.read_csv(f"{PATH_PREFIX}/output/transcriptions_translated.csv")
translations['file_name'] = translations['file_name'].str.rsplit('.', n=1).str[0]  # strip extension (e.g. "7123456789.mp4" -> "7123456789")

transcriptions_and_translations = pd.merge(videos, translations, how='left', left_on='meta_id', right_on='file_name')
transcriptions_and_translations = transcriptions_and_translations.drop(columns=['path', 'file_name'])  # drop redundant columns from the transcriptions side

transcriptions_and_translations.to_csv(f"{PATH_PREFIX}/output/transcriptions_and_translations.csv", index=False)
